# Hour 3 · Divination as decision trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/3c_divination_trees.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F3c_divination_trees.ipynb)


**Goal:** formalize an omen text's *if → then* logic as data and visualize it as a tree — then discuss where LLMs help and where they mislead.

We use a real Ugaritic **birth-omen** text and a hand-built decision tree of it (`data/omens/`).

**Needs:** `networkx`.

*Reading:* `docs/07-divination.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_omen_text, load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

## 1. The omen text (English working translation)
Note the conditional pattern: *if the foetus lacks X → consequence for land/king/cattle*.

In [ ]:
txt = load_omen_text()
print(txt[:900])

## 2. The same logic, formalized as a tree (nested JSON)

In [ ]:
from data.loader import load_omen_tree
import json
tree = load_omen_tree()
print(json.dumps(tree, indent=2, ensure_ascii=False)[:700], "...")

## 3. Turn the nested dict into a graph

In [ ]:
import networkx as nx
G = nx.DiGraph()
counter = [0]
def add(node_label, subtree, parent=None):
    nid = counter[0]; counter[0]+=1
    G.add_node(nid, label=str(node_label), leaf=not isinstance(subtree, dict))
    if parent is not None: G.add_edge(parent, nid)
    if isinstance(subtree, dict):
        for k, v in subtree.items():
            add(k, v, nid)
    else:
        lid = counter[0]; counter[0]+=1
        G.add_node(lid, label="→ "+str(subtree)[:40], leaf=True)
        G.add_edge(nid, lid)
add("omen", tree["omen"])
print("nodes:", G.number_of_nodes())

## 4. Draw the decision tree

In [ ]:
import matplotlib.pyplot as plt, collections
def layered_pos(G, root=0):
    depth={root:0}; q=collections.deque([root])
    while q:
        u=q.popleft()
        for v in G.successors(u): depth[v]=depth[u]+1; q.append(v)
    rows=collections.defaultdict(list)
    for n,d in depth.items(): rows[d].append(n)
    pos={}
    for d,ns in rows.items():
        for i,n in enumerate(ns): pos[n]=(d, -i)
    return pos
pos = layered_pos(G)
labels = {n:G.nodes[n]["label"] for n in G.nodes()}
colors = ["#ffe2b8" if not G.nodes[n]["leaf"] else "#d9f2d9" for n in G.nodes()]
plt.figure(figsize=(13,11))
nx.draw(G, pos, labels=labels, node_color=colors, node_size=900, font_size=7,
        arrows=True, edge_color="#aaa")
plt.title("Sheep-birth omen series as a decision tree")
plt.axis("off"); plt.show()

## 5. Where LLMs help — and where they are dangerous
**Help:** extract this structure from raw text, validate the JSON, compare two omen trees, generate a plain-language explanation.

**Danger:** *invent* missing branches (the text is full of `[...]` gaps), *smooth over* ambiguity, *lose* philological detail.

> **Try it (presenter):** paste the omen text into an LLM and ask for this JSON tree — then check what it filled in that the tablet does **not** say. That gap is the lesson: philology + data modelling + LLM, with a human in the loop.